In [26]:
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForMaskedLM
from transformers import get_linear_schedule_with_warmup
from peft import LoraConfig, get_peft_model, TaskType
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

torch.manual_seed(42)
np.random.seed(42)

In [27]:
# Load and examine the dataset
file_path = 'data/processed/dataset.json'

with open(file_path, 'r') as f:
    raw_data = json.load(f)

# Clean the data by removing brackets and ensuring proper format
data = []
for item in raw_data:
    cleaned_item = {
        "pro": item["pro"].replace("[", "").replace("]", "").strip(),
        "anti": item["anti"].replace("[", "").replace("]", "").strip()
    }
    data.append(cleaned_item)

print(f"Loaded {len(data)} counterfactual pairs")
print(f"Example pair:")
print(f"  Pro-stereotyped: {data[0]['pro']}")
print(f"  Anti-stereotyped: {data[0]['anti']}")

Loaded 1584 counterfactual pairs
Example pair:
  Pro-stereotyped: The developer argued with the designer because he did not like the design.
  Anti-stereotyped: The developer argued with the designer because she did not like the design.


In [28]:
model_name = "distilbert-base-uncased"
# check if mps is also available
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")

print(f"Using device: {device}")
print(f"Loading model: {model_name}")

tokenizer = AutoTokenizer.from_pretrained(model_name)
base_model = AutoModelForMaskedLM.from_pretrained(model_name)

Using device: mps
Loading model: distilbert-base-uncased


In [29]:
lora_config = LoraConfig(
    task_type=TaskType.FEATURE_EXTRACTION,
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["q_lin", "v_lin"],
    bias="none",
    modules_to_save=None,
)

model = get_peft_model(base_model, lora_config)
model = model.to(device)

model.print_trainable_parameters()

print(f"Model moved to {device}")

trainable params: 294,912 || all params: 67,280,442 || trainable%: 0.4383
Model moved to mps


In [ ]:
def debiasing_loss_fn(pro_logits, anti_logits, pro_labels, anti_labels, alpha=0.5):    
    # 1. Consistency Loss (L_consistency)
    # Measures difference between pro and anti-stereotyped logits
    consistency_loss = F.mse_loss(pro_logits, anti_logits)
    
    # 2. Task Loss (L_task) - Masked Language Modeling Loss
    # Reshape logits and labels for cross-entropy calculation
    pro_logits_flat = pro_logits.view(-1, pro_logits.size(-1))  # [batch_size * seq_len, vocab_size]
    anti_logits_flat = anti_logits.view(-1, anti_logits.size(-1))
    pro_labels_flat = pro_labels.view(-1)  # [batch_size * seq_len]
    anti_labels_flat = anti_labels.view(-1)
    
    # Calculate MLM loss for both pro and anti examples
    pro_task_loss = F.cross_entropy(pro_logits_flat, pro_labels_flat, ignore_index=-100)
    anti_task_loss = F.cross_entropy(anti_logits_flat, anti_labels_flat, ignore_index=-100)
    task_loss = (pro_task_loss + anti_task_loss) / 2
    
    # 3. Combined Loss (L_total)
    # L_total = (1 - α) * L_task + α * L_consistency
    total_loss = (1 - alpha) * task_loss + alpha * consistency_loss
    
    return {
        'total_loss': total_loss,
        'task_loss': task_loss,
        'consistency_loss': consistency_loss,
        'alpha': alpha
    }

In [31]:
class CounterfactualDataset(Dataset):
    def __init__(self, data, tokenizer, max_length=128):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        pro_text = item['pro']
        anti_text = item['anti']

        pro_encoding = self.tokenizer(
            pro_text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        anti_encoding = self.tokenizer(
            anti_text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )

        return {
            'pro_input_ids': pro_encoding['input_ids'].squeeze(),
            'pro_attention_mask': pro_encoding['attention_mask'].squeeze(),
            'anti_input_ids': anti_encoding['input_ids'].squeeze(),
            'anti_attention_mask': anti_encoding['attention_mask'].squeeze()
        }

In [33]:
# split dataset for training and testing

train, test = train_test_split(data, test_size=0.2, random_state=42)

train_dataset = CounterfactualDataset(train, tokenizer)
test_dataset = CounterfactualDataset(test, tokenizer)

train_dataloader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=16, shuffle=False)


In [34]:
epochs = 2
optimizer = AdamW(model.parameters(), lr=5e-5)
total_steps = len(train_dataloader) * epochs
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)

In [35]:
for epoch in range(epochs):
    model.train()
    epoch_loss = 0.0
    for batch in tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{epochs}"):
        pro_input_ids = batch['pro_input_ids'].to(device)
        pro_attention_mask = batch['pro_attention_mask'].to(device)
        anti_input_ids = batch['anti_input_ids'].to(device)
        anti_attention_mask = batch['anti_attention_mask'].to(device)

        # Create labels for MLM (shifted input ids)
        pro_labels = pro_input_ids.clone()
        anti_labels = anti_input_ids.clone()

        # Forward pass for pro-stereotyped input
        pro_outputs = model(input_ids=pro_input_ids, attention_mask=pro_attention_mask)
        pro_logits = pro_outputs.logits

        # Forward pass for anti-stereotyped input
        anti_outputs = model(input_ids=anti_input_ids, attention_mask=anti_attention_mask)
        anti_logits = anti_outputs.logits

        # Compute debiasing loss
        losses = debiasing_loss_fn(pro_logits, anti_logits, pro_labels, anti_labels, alpha=0.5)
        total_loss = losses['total_loss']

        # Backpropagation and optimization
        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()
        scheduler.step()

        epoch_loss += total_loss.item()

    avg_epoch_loss = epoch_loss / len(train_dataloader)
    print(f"Epoch {epoch+1}/{epochs} - Average Loss: {avg_epoch_loss:.4f}")

Epoch 1/2: 100%|██████████| 80/80 [07:06<00:00,  5.33s/it]


Epoch 1/2 - Average Loss: 7.4264


Epoch 2/2: 100%|██████████| 80/80 [04:34<00:00,  3.43s/it]

Epoch 2/2 - Average Loss: 5.8577


In [38]:
model_path = "output/debiased_model"

model.save_pretrained(model_path)

In [ ]:
from peft import PeftModel

base_model_test = AutoModelForMaskedLM.from_pretrained(model_name)

debiased_model = PeftModel.from_pretrained(base_model_test, model_path)
debiased_model = debiased_model.to(device)
debiased_model.eval()


Loading the saved debiased model...
Model loaded successfully!


In [ ]:
test_loss = 0.0
with torch.no_grad():
    for batch in tqdm(test_dataloader, desc="Evaluating"):
        pro_input_ids = batch['pro_input_ids'].to(device)
        pro_attention_mask = batch['pro_attention_mask'].to(device)
        anti_input_ids = batch['anti_input_ids'].to(device)
        anti_attention_mask = batch['anti_attention_mask'].to(device)

        # Create labels for MLM (shifted input ids)
        pro_labels = pro_input_ids.clone()
        anti_labels = anti_input_ids.clone()

        # Forward pass for pro-stereotyped input
        pro_outputs = debiased_model(input_ids=pro_input_ids, attention_mask=pro_attention_mask)
        pro_logits = pro_outputs.logits

        # Forward pass for anti-stereotyped input
        anti_outputs = debiased_model(input_ids=anti_input_ids, attention_mask=anti_attention_mask)
        anti_logits = anti_outputs.logits

        # Compute debiasing loss
        losses = debiasing_loss_fn(pro_logits, anti_logits, pro_labels, anti_labels, alpha=0.5)
        total_loss = losses['total_loss']

        test_loss += total_loss.item()
    
avg_test_loss = test_loss / len(test_dataloader)
print(f"Test Set - Average Loss: {avg_test_loss:.4f}")

Evaluating: 100%|██████████| 20/20 [00:41<00:00,  2.06s/it]

Test Set - Average Loss: 5.3733
